In [2]:
!pip install pandas openpyxl biopython requests rapidfuzz pymupdf pdfplumber openai tqdm

  You can safely remove it manually.


  Using cached biopython-1.87-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached rapidfuzz-3.14.5-cp313-cp313-win_amd64.whl.metadata (12 kB)
  Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl.metadata (24 kB)
  Using cached jiter-0.15.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
Using cached biopython-1.87-cp313-cp313-win_amd64.whl (2.7 MB)
Using cached rapidfuzz-3.14.5-cp313-cp313-win_amd64.whl (1.5 MB)
Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl (19.2 MB)
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.6 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.6 MB 376.4 kB/s eta 0:00:17
   ---- ----------------------------------- 0.8/6.6 MB 573.4 kB/s eta 0:00:11
   ------ --------------------------------- 1.0/6.6 MB 707.9 kB/s eta 0:00:08
   --------- ------------------------------ 1.6/6.6 MB 974.7 kB/s eta 0:

In [1]:
import os
import pandas as pd
from Bio import Entrez
from rapidfuzz import fuzz
import ollama
from tqdm import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
Entrez.email = "ajayitemitopejeremiah@gmail.com"  # Replace with your email
SEARCH_QUERY = "machine learning AND thyroid cancer diagnosis"
MAX_RESULTS = 50  # Adjust based on your needs

INCLUSION_CRITERIA = """
1. The study must use Machine Learning or Deep Learning techniques.
2. The study must focus on diagnosing thyroid cancer.
3. The study must use human ultrasound images.
Exclude any general reviews, editorials, or animal-only trials.
"""

# ==========================================
# STEP 1: FETCH DATA FROM PUBMED
# ==========================================
def fetch_pubmed_papers(query, max_results):
    print(f"🔍 Searching PubMed for: '{query}'...")
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    record = Entrez.read(handle)
    handle.close()
    id_list = record["IdList"]
    
    if not id_list:
        print("❌ No papers found for this query.")
        return pd.DataFrame()

    print(f"📥 Fetching metadata for {len(id_list)} papers...")
    handle = Entrez.efetch(db="pubmed", id=",".join(id_list), retmode="xml")
    records = Entrez.read(handle)
    handle.close()
    
    articles = []
    for article in records['PubmedArticle']:
        try:
            title = article['MedlineCitation']['Article']['ArticleTitle']
            
            # Extract Abstract safely
            abstract_text = article['MedlineCitation']['Article']['Abstract']['AbstractText']
            abstract = " ".join(abstract_text) if isinstance(abstract_text, list) else str(abstract_text)
            
            # Extract DOI safely
            doi = "N/A"
            for id_elem in article['PubmedArticle']['PubmedData']['ArticleIdList']:
                if id_elem.attributes.get('IdType') == 'doi':
                    doi = str(id_elem)
                    break
                    
            articles.append({"Title": title, "Abstract": abstract, "DOI": doi})
        except (KeyError, IndexError):
            continue
            
    return pd.DataFrame(articles)

# ==========================================
# STEP 2: SMART DEDUPLICATION
# ==========================================
def deduplicate_papers(df, similarity_threshold=90):
    print("🧹 Starting deduplication process...")
    initial_count = len(df)
    
    # Remove exact DOI matches
    df = df.drop_duplicates(subset=['DOI'])
    
    # Remove fuzzy title matches
    titles = df['Title'].tolist()
    indices_to_drop = []
    
    for i in range(len(titles)):
        for j in range(i + 1, len(titles)):
            if fuzz.ratio(titles[i].lower(), titles[j].lower()) > similarity_threshold:
                indices_to_drop.append(df.index[j])
                
    df_cleaned = df.drop(index=list(set(indices_to_drop)))
    print(f"✅ Deduplication finished. Removed {initial_count - len(df_cleaned)} duplicates.")
    return df_cleaned

# ==========================================
# STEP 3: AI SCREENING (LOCAL OLLAMA LLM)
# ==========================================
def screen_abstract_with_ai(title, abstract, criteria):
    prompt = f"""
    You are an expert medical researcher conducting a Systematic Literature Review.
    
    Strict Eligibility Criteria:
    {criteria}
    
    Evaluate this paper:
    Title: {title}
    Abstract: {abstract}
    
    Does this paper satisfy all inclusion criteria? 
    Respond with exactly ONE word: either 'INCLUDE' or 'EXCLUDE'. Do not output anything else.
    """
    try:
        response = ollama.generate(model='llama3', prompt=prompt)
        decision = response['response'].strip().upper()
        # Clean up any unexpected LLM conversational text
        if "INCLUDE" in decision:
            return "INCLUDE"
        return "EXCLUDE"
    except Exception as e:
        return f"ERROR ({str(e)})"

In [4]:
import sys
!{sys.executable} -m pip install ollama

  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
